# Tokenization

Tokenization is a way of solving a basic problem of working with language data. In Python, we're used to dealing with language (say, an English sentence) as strings:

In [1]:
sentence = "That's very interesting!"

Language models don't work with strings directly. If we think of the simple neural models we've covered so far, they all deal in floating point numbers. So how do we go from this string representation of language (sequence of symbols) to a numerical one?

Let's start with a simpler question. If I asked you how many words are in the sentence above, what would you say? 

In [2]:
num_words = len(sentence.split())
print(num_words) 

3


If we split on whitespace, that gives us three words: 

In [3]:
words = sentence.split()
print(words)

["That's", 'very', 'interesting!']


If we have many sentences, we can do this to all of them to build a *vocabulary*: a finite set of words that we (or our model) knows about. Let's use these English and French sentences as an example:

In [4]:
sentences = [
    "That's very interesting!",
    "I didn't know that.",
    "C'est très interessant.", 
    "Je ne le savais pas."
]

vocab = list({word for s in sentences for word in s.split()})
vocab

['very',
 'Je',
 "That's",
 "didn't",
 'ne',
 'interessant.',
 "C'est",
 'I',
 'that.',
 'interesting!',
 'le',
 'know',
 'très',
 'pas.',
 'savais']

This gives us a vocabulary of all of the unique words in our corpus (list of sentences). However, just tokenizing on whitespace has a few obvious limitations: 
- Words with contractions ("didn't", "That's") occupy a distinct place in the vocab from the words that compose them ("did", "not", "that", "is"). 
- Punctuation marks are "attached" to preceding words ("interesting!")
- Should we care about case ("That's" vs "that's")?

If we wanted a more robust whitespace-ish tokenizer, we might do something like this:

In [5]:
import re

pattern = re.compile(r"\w+|[^\w\s]+")

def pretokenizer(s):
    return re.findall(pattern, s.lower())

pretokenized = [pretokenizer(s) for s in sentences]

vocab = list({w for s in pretokenized for w in s})
print(vocab)

['that', 'interesting', '!', 'ne', 'pas', 'didn', 'le', 'savais', 'very', 'c', '.', 's', 'i', 'je', 'interessant', 'est', "'", 't', 'know', 'très']


Ok, that's a little better: punctuation is tokenized separately from other text and we lowercase everything. But now, "didn't" becomes "didn" + "'" + "t", which is arguably worse than just leaving it alone. And if we were to tokenize another French word like "aujourd'hui" ("today"), it would split into "aujourd" and "hui", even though neither of those are words on their own!

For now, though, let's just say that this is good enough. Let's build a simple tokenizer that maps substrings (words, punctuation) to *integer IDs*:



In [6]:
token2id = {w: i for i, w in enumerate(vocab)}
id2token = {v: k for k, v in token2id.items()}

def tokenizer(s):
    s = pretokenizer(s)
    return [token2id[w] for w in s]

def detokenizer(ids):
    return [id2token[i] for i in ids]

tokenized = tokenizer("That's interesting.")
detokenized = detokenizer(tokenized)
print(tokenized)
print(detokenized)


[0, 16, 11, 1, 10]
['that', "'", 's', 'interesting', '.']


At a very general level, this is what a tokenizer in an LLM does: it separates longer strings into shorter substrings (pretokenization), then maps those substrings to integer IDs. It is these IDs which the model "sees". 

But our tokenizer has a massive limitation that LLM tokenizers don't. Let's try this:

In [7]:
tokenizer("Is that very interesting?")

KeyError: 'is'

We have no way of dealing with words we haven't seen before. When we go to find "is" in our vocabulary, there's nothing there, so we get a `KeyError`. Words that are not in our vocabulary are called "out-of-vocabulary" or OOV. One way of handling OOV words is to use an "UNK" token:

In [8]:
unk_id = len(id2token) + 1
id2token[unk_id] = "UNK"
token2id["UNK"] = unk_id

def better_tokenizer(s):
    s = pretokenizer(s)
    return [token2id[w] 
            if w in token2id else token2id["UNK"]
            for w in s]

tokenized = better_tokenizer("Is that very interesting?")
detokenized = detokenizer(tokenized)

In [9]:
print(tokenized)
print(detokenized)

[21, 0, 8, 1, 21]
['UNK', 'that', 'very', 'interesting', 'UNK']


If our vocabulary is huge, we might be fine with this approach. But it's not very elegant, since we're treating all unknown words the exact same, and it poses some problems for training--we need the model to see some UNKs so that it knows what to do with them at inference time, but no training token is actually an UNK. 

UNK tokens keep us from getting an error message, but they don't really solve the problem: we want a fixed vocabulary that can represent all (or at least nearly all) words we might ever encounter at inference time. But what would that look like?

### Comparing tokenizers and Byte-level Byte Pair Encoding
Today's lecture looks at real model tokenizers: what they do and how they actually work. We'll see how these tokenizers solve this OOV problem while still keeping vocabularies relatively small!

# Compare Tokenizers

Different language models are trained with different tokenizers, which determine how text is broken into tokens. The model continues to use that same tokenizer after training to process new text, so that words and symbols are represented in the same way the model learned during training. These tokenization strategies can vary in how they handle punctuation, whitespace, subwords, and even entire words. Because of this, the choice of tokenizer can influence how efficiently a model represents text and how well it performs on certain types of language or tasks.

In [10]:
from transformers import AutoTokenizer
from IPython.display import HTML

 In the code cell below, we load the tokenizer associated with the GPT-2 (`gpt2`) model.

In [20]:
model_name = "intfloat/multilingual-e5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

We can identify the vocabulary size of the tokenizer and print it.

In [21]:
vocab_size = tokenizer.vocab_size
print(vocab_size)

250002


When decoding token IDs back into text, some tokens &mdash; like spaces, tabs, or newlines &mdash; are invisible in normal output. To make them easier to see, we’ll defind a helper function called `make_printable()` that takes a decoded token string and replaces those invisible characters with visible symbols such as ␣ for spaces, ↵ for newlines, and → for tabs.

In [22]:
def make_printable(s):
    replacements = {
        ' ': '␣',
        '\n': '↵',
        '\r': '↵',
        '\t': '→'
    }
    result = s
    for old, new in replacements.items():
        result = result.replace(old, new)
    
    # Replace non-printable characters like control sequences
    result = ''.join(c if c.isprintable() or c in replacements else f'\\x{ord(c):02x}' for c in result)
    return result

In the code cell below, we call the `make_printable()` function several times with different decoded token strings. First, use the tokenizer to decode a token ID into text, and then pass the resulting string to `make_printable()` to display any invisible characters as visible symbols.

In [23]:
print(make_printable(tokenizer.decode(10)))
print(make_printable(tokenizer.decode(100)))
print(make_printable(tokenizer.decode(1000)))
print(make_printable(tokenizer.decode(10000)))

a
for
trong
SI


The `display_tokens()` function defined below displays a range of tokens in a dense HTML format, meaning they appear in a tight, visually organized grid rather than as plain text lines, so you can visually explore the tokenizer’s vocabulary.

The function takes a starting token ID (`start`) and an optional number of tokens to show (`display_count` - which has a default value of 500).
It then displays that range of tokens — from `start` up to `start + display_count` — in a compact HTML grid.

In [24]:
def display_tokens(start, display_count=500, tokenizer=tokenizer):
    # Build HTML
    html_parts = ['<div style="font-family: monospace; line-height: 1.6; font-size: 13px;">']
    
    # Two alternating light gray colors
    colors = ['#e8e8e8', '#f5f5f5']
    
    for token_id in range(start, start+display_count):
        # Decode the token ID to get the string representation
        token_str = tokenizer.decode([token_id])
        printable_token = make_printable(token_str)
        
        # Escape HTML special characters
        printable_token = (printable_token
                          .replace('&', '&amp;')
                          .replace('<', '&lt;')
                          .replace('>', '&gt;')
                          .replace('"', '&quot;'))
        
        bg_color = colors[token_id % 2]
        
        html_parts.append(
            f'<span style="background-color: {bg_color}; '
            f'padding: 2px 4px; margin: 1px; display: inline-block; '
            f'border-radius: 3px; border: 1px solid rgba(0,0,0,0.1);" '
            f'title="ID: {token_id}">{printable_token}</span>'
        )
    
    html_parts.append('</div>')
    
    # Display
    return HTML(''.join(html_parts))

In the code cells below, we call the function `display_tokens()` several times with different starting positions (different values for `start`), including:

* 0 (the beginning)
* A value in the low thousands
* One in the middle of the vocabulary
* One near the end

Tokens that do not map to valid UTF-8 characters (because they contain unfinished byte sequences) will be shown as �. Tokens that are valid but cannot be displayed will appear as their byte sequences.

In [25]:
display_tokens(0)

In [26]:
display_tokens(1000)

In [27]:
display_tokens(25000)

In [28]:
display_tokens(50000)

Now, let's choose a language model other than GPT-2 from the Hugging Face hub. We'll set the variable `model_name` that we defined earlier to the identifier string for that model and re-run the cells in this notebook to use the tokenizer for the new model. 

What was different between the tokenizers? What was the same?

# Byte-Level Byte Pair Encoding (BPE)

But how do these LLM tokenizers actually work? How were they trained? 

The  byte-pair encoding (BPE) algorithm was used to develop the tokenizer for GPT-2. The goal of the BPE algorithm is to identify a set of **tokens** that efficiently represent the text in a training corpus. It begins by splitting text into individual bytes (the smallest units of digital text, representing characters, spaces, and punctuation), initializing a base vocabulary that includes every possible byte, and then iteratively merges the most frequent pairs of tokens to form new ones.

This means that at the start, the algorithm treats every byte (character, space, or punctuation mark) as its own token.
It builds an initial vocabulary containing all possible bytes so that any text can be represented.
Then it repeatedly looks for the most common pair of tokens &mdash; for example, “t” followed by “h” &mdash; and merges them into a single new token (“th”).
By repeating this process many times, the tokenizer learns to represent frequent patterns such as “the” or “ing” as single tokens, creating a compact and efficient vocabulary.

>**Note**: The term “byte” in BPE comes from the algorithm’s origin in data compression, where it merged frequently occurring pairs of bytes to reduce file size. In natural language processing, the same idea is adapted to merge frequent pairs of characters or subword units. Models like GPT-2 use a byte-level version of BPE so that any text — including emojis and special characters — can be represented consistently without language-specific preprocessing. BERT, another language model, uses SentencePiece, which is a similar algorithm.

We’ll apply BPE to the poem “Song of the Jellicles” by T. S. Eliot, which contains several repeated phrases. Because BPE works by finding frequent pairs of tokens and merging them to create new tokens that efficiently represent recurring text patterns, the poem’s repetition makes it a strong example for observing token merges. 

In [30]:
song_of_the_jellicles = """
Jellicle Cats come out tonight,
Jellicle Cats come one come all:
The Jellicle Moon is shining bright—
Jellicles come to the Jellicle Ball.

Jellicle Cats are black and white,
Jellicle Cats are rather small;
Jellicle Cats are merry and bright,
And pleasant to hear when they caterwaul.
Jellicle Cats have cheerful faces,
Jellicle Cats have bright black eyes;
They like to practise their airs and graces
And wait for the Jellicle Moon to rise.

Jellicle Cats develop slowly,
Jellicle Cats are not too big;
Jellicle Cats are roly-poly,
They know how to dance a gavotte and a jig.
Until the Jellicle Moon appears
They make their toilette and take their repose:
Jellicles wash behind their ears,
Jellicles dry between their toes.

Jellicle Cats are white and black,
Jellicle Cats are of moderate size;
Jellicles jump like a jumping-jack,
Jellicle Cats have moonlit eyes.
They're quiet enough in the morning hours,
They're quiet enough in the afternoon,
Reserving their terpsichorean powers
To dance by the light of the Jellicle Moon.

Jellicle Cats are black and white,
Jellicle Cats (as I said) are small;
If it happens to be a stormy night
They will practise a caper or two in the hall.
If it happens the sun is shining bright
You would say they had nothing to do at all:
They are resting and saving themselves to be right
For the Jellicle Moon and the Jellicle Ball.
"""

What full-word tokens do you think might occur early in the merging process? Why?

## Creating a New Tokenizer using BPE

We can create a new tokenizer based on an existing one by using the `train_new_from_iterator()` function. 

In the code cell below, we define the GPT-2 tokenizer as the base tokenizer. It was developed using the BPE algorithm and follows byte-level tokenization rules. Using the `train_new_from_iterator()` function, we can create a new tokenizer from this base, applying the same byte-level rules while learning a vocabulary of a specified size from the provided text. The first argument is an iterable of text strings (in this case, a list containing the poem “Song of the Jellicles”), and the second argument specifies the total number of distinct tokens to include in the new vocabulary — here, 1,000 tokens learned from the most frequent patterns in the text through the iterative merging process described above.

In [31]:
base_tokenizer = AutoTokenizer.from_pretrained("gpt2")
new_tokenizer = base_tokenizer.train_new_from_iterator([song_of_the_jellicles], 1000)

The code cell below uses the `decode()` function on each token ID in the tokenizer’s vocabulary to produce a list of all token strings. This allows us to view the full set of tokens that make up the tokenizer’s learned vocabulary.

In [32]:
vocabulary = [ new_tokenizer.decode(i) for i in range(new_tokenizer.vocab_size) ]

The code cell below prints the first 500 tokens in the vocabulary. The first 256 tokens are seeded by the algorithm — they represent all possible single bytes, many of which cannot be displayed as valid UTF-8 characters. Scroll down past token 257 to see the first *merged* tokens created by the BPE process. 

In [33]:
for i, token in enumerate(vocabulary[:500]):
    print(f"{i}\t'{token}'")

0	'<|endoftext|>'
1	'!'
2	'"'
3	'#'
4	'$'
5	'%'
6	'&'
7	'''
8	'('
9	')'
10	'*'
11	'+'
12	','
13	'-'
14	'.'
15	'/'
16	'0'
17	'1'
18	'2'
19	'3'
20	'4'
21	'5'
22	'6'
23	'7'
24	'8'
25	'9'
26	':'
27	';'
28	'<'
29	'='
30	'>'
31	'?'
32	'@'
33	'A'
34	'B'
35	'C'
36	'D'
37	'E'
38	'F'
39	'G'
40	'H'
41	'I'
42	'J'
43	'K'
44	'L'
45	'M'
46	'N'
47	'O'
48	'P'
49	'Q'
50	'R'
51	'S'
52	'T'
53	'U'
54	'V'
55	'W'
56	'X'
57	'Y'
58	'Z'
59	'['
60	'\'
61	']'
62	'^'
63	'_'
64	'`'
65	'a'
66	'b'
67	'c'
68	'd'
69	'e'
70	'f'
71	'g'
72	'h'
73	'i'
74	'j'
75	'k'
76	'l'
77	'm'
78	'n'
79	'o'
80	'p'
81	'q'
82	'r'
83	's'
84	't'
85	'u'
86	'v'
87	'w'
88	'x'
89	'y'
90	'z'
91	'{'
92	'|'
93	'}'
94	'~'
95	'�'
96	'�'
97	'�'
98	'�'
99	'�'
100	'�'
101	'�'
102	'�'
103	'�'
104	'�'
105	'�'
106	'�'
107	'�'
108	'�'
109	'�'
110	'�'
111	'�'
112	'�'
113	'�'
114	'�'
115	'�'
116	'�'
117	'�'
118	'�'
119	'�'
120	'�'
121	'�'
122	'�'
123	'�'
124	'�'
125	'�'
126	'�'
127	'�'
128	'�'
129	'�'
130	'�'
131	'�'
132	'�'
133	'�'
134	'�'
135	'�'
136	'�'
13

## Comparing our Tokenizer to the Original

The GPT-2 tokenizer was created from millions of internet documents, allowing it to capture a broad and diverse vocabulary. The new tokenizer has only seen *Song of the Jellicles*. We can use the function `get_tokens()` below to tokenize the phrase "The Jellicle Moon is shining bright" using both tokenizers and compare the two tokenizers.

In [34]:
def get_tokens(s, tokenizer):
    return tokenizer.convert_ids_to_tokens(tokenizer(s)["input_ids"])

In [35]:
print("Our tokenizer: ", get_tokens("The Jellicle Moon is shining bright", new_tokenizer))
print("Original tokenizer: ", get_tokens("The Jellicle Moon is shining bright", base_tokenizer))

Our tokenizer:  ['The', 'ĠJellicle', 'ĠMoon', 'Ġis', 'Ġshining', 'Ġbright']
Original tokenizer:  ['The', 'ĠJ', 'ell', 'icle', 'ĠMoon', 'Ġis', 'Ġshining', 'Ġbright']


Let's look at a short phrase that is not part of the poem and in the code cell below, tokenize it using both tokenizers.

In [36]:
print("Our tokenizer: ", get_tokens("William the Conqueror's forces defeated the English army at Hastings and killed Harold Godwinson", new_tokenizer))
print("Original tokenizer: ", get_tokens("William the Conqueror's forces defeated the English army at Hastings and killed Harold Godwinson", base_tokenizer))

Our tokenizer:  ['W', 'ill', 'i', 'a', 'm', 'Ġthe', 'Ġ', 'C', 'on', 'qu', 'er', 'or', "'", 's', 'Ġfor', 'c', 'es', 'Ġd', 'e', 'f', 'e', 'ate', 'd', 'Ġthe', 'Ġ', 'E', 'n', 'g', 'l', 'is', 'h', 'Ġa', 'r', 'my', 'Ġat', 'Ġ', 'H', 'as', 't', 'ing', 's', 'Ġand', 'Ġ', 'k', 'ill', 'e', 'd', 'Ġ', 'H', 'ar', 'o', 'ld', 'Ġ', 'G', 'o', 'd', 'w', 'in', 's', 'on']
Original tokenizer:  ['William', 'Ġthe', 'ĠConquer', 'or', "'s", 'Ġforces', 'Ġdefeated', 'Ġthe', 'ĠEnglish', 'Ġarmy', 'Ġat', 'ĠHastings', 'Ġand', 'Ġkilled', 'ĠHarold', 'ĠGod', 'w', 'inson']


## Why not just use bytes? 

The above example is pretty extreme: we only train our tokenizer on a single text, so it is obviously overfit and will generalize poorly to unseen words (even if it doesn't "UNK"). But if you think about it, why bother to build a huge vocabulary (with 50k subwords!) when we could just use bytes? If we did that, we'd just have 256 entries in our vocab, and we could still encode anything we wanted! 

In [37]:
def get_bytes(s):
    """Get the unicode point of a character."""
    return [b for c in s for b in c.encode()]

In [38]:
print(get_bytes("Hello"))

[72, 101, 108, 108, 111]


In [39]:
print(get_bytes("你好"))

[228, 189, 160, 229, 165, 189]


In [40]:
print(get_bytes("السلام عليكم"))

[216, 167, 217, 132, 216, 179, 217, 132, 216, 167, 217, 133, 32, 216, 185, 217, 132, 217, 138, 217, 131, 217, 133]


In [41]:
print(get_bytes("გამარჯობა"))

[225, 131, 146, 225, 131, 144, 225, 131, 155, 225, 131, 144, 225, 131, 160, 225, 131, 175, 225, 131, 157, 225, 131, 145, 225, 131, 144]


Some language models do actually use bytes instead of subword tokens (ByT5, BOLMO). However, it's not all upside:

In [42]:
example_sentence = "All human beings are born free and equal in dignity and rights."
print("Length (tokens, Jellicle tokenizer): ", len(get_tokens(example_sentence, new_tokenizer)))
print("Length (tokens, GPT-2 tokenizer): ", len(get_tokens(example_sentence, base_tokenizer)))
print("Length (bytes): ", len(get_bytes(example_sentence)))

Length (tokens, Jellicle tokenizer):  31
Length (tokens, GPT-2 tokenizer):  13
Length (bytes):  63


Using bytes to encode our sequence results in double the tokens as our toy Jellicle tokenizer, and almost six times as many as if we had just used GPT-2's! The point of BPE isn't just to get around the OOV problem: we want to encode our inputs efficiently. This is especially important for a transformer model, since the computational cost of processing a sequence grows quadratically with the number of tokens in the input sequence. Generally speaking, don't mind having a larger vocab if it means that we can fit more information into fewer input tokens. 

### A brief note on noise
A known issue with tokenizers like the one GPT-2 uses is that they aren't very robust to noise:

In [43]:
print("Original: ", get_tokens("It's almost lunchtime!", base_tokenizer))
print("Misspelling: ", get_tokens("It's amlost lunchtime!", base_tokenizer))

Original:  ['It', "'s", 'Ġalmost', 'Ġlunch', 'time', '!']
Misspelling:  ['It', "'s", 'Ġam', 'lost', 'Ġlunch', 'time', '!']


For a human being, this misspelling is just a single character swap that we can easily "repair". For the model, however, this turns a single ID into two completely different ones. I've [written a paper](https://www.semanticscholar.org/paper/Semantics-or-spelling-Probing-contextual-word-with-Matthews-Starr/ea23d4dbb1391e3c61717b0ddb915dcb79ad499f) where we investigate this problem more broadly. Even though BPE splits words into subword tokens, models like GPT-2 were trained on mostly English data, and so most English words are tokenized with a single token. As a result, misspellings and other forms of orthographic noise can have an outsized impact on model representations downstream. 

(This also partially explains why even large models have trouble knowing how words are spelled, like "How many rs are in strawberry?". Why do you think this problem so hard?)

### Special Characters and Padding
Finally, tokenizers perform another crucial job for an LLM. They add "special" tokens (e.g. signalling the beginning/end of a sequence) and "pad" multiple sequences in a batch to a consistent length. We can inspect these special tokens like so:

In [ ]:
base_tokenizer.special_tokens_map

In [ ]:
base_tokenizer.pad_token_type_id

We need to pad our sequence because an input tensor can't be ragged. A language model can process multiple sequences in parallel (a batch), but each of these sequences needs to be the same length. The tokenizer takes the batch, identifies the longest sequence, then "pads" all the other sequences to be exactly as long as the longest sequence using a padding token that the model ignores (we'll talk more about how that works next week). 